In [2]:
import numpy as np
import random
import copy
import time


# ==========================
# Instância do Sudoku
# ==========================

sudoku = np.array([
    [6, 0, 0, 0, 8, 0, 0, 0, 0],
    [0, 0, 9, 7, 0, 0, 0, 2, 8],
    [2, 0, 0, 0, 0, 0, 3, 0, 0],

    [0, 0, 0, 0, 0, 0, 0, 8, 0],
    [0, 0, 0, 4, 5, 0, 0, 9, 0],
    [0, 9, 3, 0, 2, 0, 0, 0, 1],

    [0, 0, 0, 6, 0, 0, 2, 0, 0],
    [4, 0, 6, 0, 0, 9, 0, 0, 0],
    [0, 0, 2, 3, 0, 0, 0, 0, 4]
])

# =====================================
# Controle de avaliações
# =====================================

MAX_AVALIACOES = 400000
AVALIACOES = 0

# ==========================
# Informações da instância
# ==========================

# Posições originalmente preenchidas
posicoes_fixas = list(zip(*np.where(sudoku != 0)))

# Posições vazias
posicoes_vazias = list(zip(*np.where(sudoku == 0)))

# Número de variáveis de decisão
N = len(posicoes_vazias)

print(f"Células fixas : {len(posicoes_fixas)}")
print(f"Células vazias: {N}")

Células fixas : 24
Células vazias: 57


In [ ]:
# =====================================
# Função de Aptidão
# =====================================

def contar_conflitos_pares(valores):
    """
    Conta todos os pares de valores iguais em um vetor.

    Exemplo:
    [5, 5, 5] possui 3 pares conflitantes:
    (5_1, 5_2), (5_1, 5_3) e (5_2, 5_3).
    """

    valores = valores[valores != 0]
    conflitos = 0

    for j in range(len(valores) - 1):
        for l in range(j + 1, len(valores)):
            if valores[j] == valores[l]:
                conflitos += 1

    return conflitos


def calcular_fitness(tabuleiro):
    """
    Calcula a quantidade total de conflitos
    nas linhas, colunas e blocos 3x3.

    Cada par de valores iguais é contabilizado
    como um conflito.

    Quanto menor o fitness, melhor a solução.
    O valor ótimo é fitness igual a zero.

    Cada chamada da função corresponde
    a uma avaliação da função objetivo.
    """
    global AVALIACOES

    # Conta uma avaliação da função objetivo
    AVALIACOES += 1

    tabuleiro = np.asarray(tabuleiro)

    if tabuleiro.shape != (9, 9):
        raise ValueError("O tabuleiro deve possuir dimensão 9x9.")

    conflitos = 0

    # ---------- Linhas ----------
    for linha in tabuleiro:
        conflitos += contar_conflitos_pares(linha)

    # ---------- Colunas ----------
    for coluna in tabuleiro.T:
        conflitos += contar_conflitos_pares(coluna)

    # ---------- Blocos 3x3 ----------
    for i in range(0, 9, 3):
        for j in range(0, 9, 3):
            bloco = tabuleiro[i:i + 3, j:j + 3].flatten()
            conflitos += contar_conflitos_pares(bloco)

    return conflitos


# =====================================
# Funções auxiliares
# =====================================

def copiar_tabuleiro(tabuleiro):
    """
    Retorna uma cópia independente do tabuleiro.
    """

    return copy.deepcopy(tabuleiro)


def vetor_para_tabuleiro(vetor, sudoku_original):
    """
    Converte um vetor de decisão em um tabuleiro de Sudoku.

    Cada valor do vetor é inserido em uma posição
    originalmente vazia do Sudoku.
    """

    if len(vetor) != len(posicoes_vazias):
        raise ValueError(
            "O tamanho do vetor deve ser igual ao número de posições vazias."
        )

    tabuleiro = sudoku_original.copy()

    for valor, (i, j) in zip(vetor, posicoes_vazias):
        tabuleiro[i, j] = valor

    return tabuleiro


def mostrar_tabuleiro(tabuleiro):
    """
    Exibe o tabuleiro de forma organizada.
    """

    for i in range(9):
        if i > 0 and i % 3 == 0:
            print("-" * 21)

        linha_formatada = []

        for j in range(9):
            if j > 0 and j % 3 == 0:
                linha_formatada.append("|")

            linha_formatada.append(str(tabuleiro[i, j]))

        print(" ".join(linha_formatada))


# =====================================
# Teste da função de aptidão
# =====================================

print("Fitness inicial:", calcular_fitness(sudoku))
print("\nTabuleiro inicial:\n")

mostrar_tabuleiro(sudoku)

Fitness inicial: 0

Tabuleiro inicial:

6 0 0 | 0 8 0 | 0 0 0
0 0 9 | 7 0 0 | 0 2 8
2 0 0 | 0 0 0 | 3 0 0
---------------------
0 0 0 | 0 0 0 | 0 8 0
0 0 0 | 4 5 0 | 0 9 0
0 9 3 | 0 2 0 | 0 0 1
---------------------
0 0 0 | 6 0 0 | 2 0 0
4 0 6 | 0 0 9 | 0 0 0
0 0 2 | 3 0 0 | 0 0 4


In [ ]:
# ============================================================
# CÉLULA 4 — ALGORITMO GENÉTICO (GA)
# ============================================================


def obter_posicoes_vazias_por_bloco(sudoku_original):
    """
    Retorna as posições originalmente vazias de cada uma das
    nove subgrades 3x3.

    O resultado é uma lista com nove elementos.
    Cada elemento contém as posições modificáveis de um bloco.
    """

    posicoes_por_bloco = []

    for inicio_linha in range(0, 9, 3):
        for inicio_coluna in range(0, 9, 3):

            posicoes_bloco = []

            for i in range(inicio_linha, inicio_linha + 3):
                for j in range(inicio_coluna, inicio_coluna + 3):

                    if sudoku_original[i, j] == 0:
                        posicoes_bloco.append((i, j))

            posicoes_por_bloco.append(posicoes_bloco)

    return posicoes_por_bloco


def gerar_individuo_ga(sudoku_original):
    """
    Gera um cromossomo mantendo todas as subgrades 3x3 válidas.

    Para cada bloco:
    1. identifica os valores fixos;
    2. determina quais números de 1 a 9 estão ausentes;
    3. embaralha os valores ausentes;
    4. distribui esses valores nas células originalmente vazias.

    Assim, cada bloco contém os números de 1 a 9 sem repetição.
    """

    individuo = sudoku_original.copy()

    for inicio_linha in range(0, 9, 3):
        for inicio_coluna in range(0, 9, 3):

            bloco_original = sudoku_original[
                inicio_linha:inicio_linha + 3,
                inicio_coluna:inicio_coluna + 3
            ]

            valores_fixos = set(
                bloco_original[bloco_original != 0].tolist()
            )

            valores_ausentes = [
                valor
                for valor in range(1, 10)
                if valor not in valores_fixos
            ]

            random.shuffle(valores_ausentes)

            posicoes_bloco = []

            for i in range(inicio_linha, inicio_linha + 3):
                for j in range(inicio_coluna, inicio_coluna + 3):

                    if sudoku_original[i, j] == 0:
                        posicoes_bloco.append((i, j))

            for (i, j), valor in zip(
                posicoes_bloco,
                valores_ausentes
            ):
                individuo[i, j] = valor

    return individuo


def gerar_populacao_inicial_ga(
    sudoku_original,
    tamanho_populacao
):
    """
    Gera a população inicial do GA.
    """

    populacao = [
        gerar_individuo_ga(sudoku_original)
        for _ in range(tamanho_populacao)
    ]

    return populacao


def avaliar_populacao_ga(populacao):
    """
    Calcula o fitness de todos os cromossomos da população.

    Como o problema é de minimização, menores valores indicam
    soluções de melhor qualidade.
    """

    fitness_populacao = np.array([
        calcular_fitness(individuo)
        for individuo in populacao
    ])

    return fitness_populacao


def selecao_torneio_ga(
    populacao,
    fitness_populacao,
    tamanho_torneio
):
    """
    Realiza a seleção pelo método K-Tournament.

    São escolhidos k cromossomos aleatoriamente.
    O cromossomo com menor fitness é selecionado como pai.
    """

    tamanho_torneio = min(
        tamanho_torneio,
        len(populacao)
    )

    indices_torneio = random.sample(
        range(len(populacao)),
        tamanho_torneio
    )

    melhor_indice = min(
        indices_torneio,
        key=lambda indice: fitness_populacao[indice]
    )

    return populacao[melhor_indice].copy()


def cruzamento_por_blocos_ga(pai_1, pai_2):
    """
    Executa o cruzamento considerando cada subgrade 3x3
    como uma unidade genética.

    Para cada um dos nove blocos, o descendente herda
    aleatoriamente o bloco do pai 1 ou do pai 2.
    """

    descendente = np.empty((9, 9), dtype=int)

    for inicio_linha in range(0, 9, 3):
        for inicio_coluna in range(0, 9, 3):

            if random.random() < 0.5:
                pai_escolhido = pai_1
            else:
                pai_escolhido = pai_2

            descendente[
                inicio_linha:inicio_linha + 3,
                inicio_coluna:inicio_coluna + 3
            ] = pai_escolhido[
                inicio_linha:inicio_linha + 3,
                inicio_coluna:inicio_coluna + 3
            ]

    return descendente


def mutacao_por_troca_ga(
    individuo,
    sudoku_original,
    probabilidade_mutacao
):
    """
    Aplica mutação com determinada probabilidade.

    A mutação consiste na troca entre duas células
    originalmente vazias pertencentes à mesma subgrade.
    """

    mutante = individuo.copy()

    if random.random() >= probabilidade_mutacao:
        return mutante

    posicoes_por_bloco = obter_posicoes_vazias_por_bloco(
        sudoku_original
    )

    blocos_mutaveis = [
        posicoes
        for posicoes in posicoes_por_bloco
        if len(posicoes) >= 2
    ]

    if not blocos_mutaveis:
        return mutante

    posicoes_bloco = random.choice(blocos_mutaveis)

    posicao_1, posicao_2 = random.sample(
        posicoes_bloco,
        2
    )

    i1, j1 = posicao_1
    i2, j2 = posicao_2

    mutante[i1, j1], mutante[i2, j2] = (
        mutante[i2, j2],
        mutante[i1, j1]
    )

    return mutante


def selecionar_melhores_candidatos_ga(
    candidatos,
    quantidade
):
    """
    Seleciona os cromossomos com menores valores de fitness
    entre uma lista de candidatos.
    """

    candidatos_ordenados = sorted(
        candidatos,
        key=calcular_fitness
    )

    return [
        candidato.copy()
        for candidato in candidatos_ordenados[:quantidade]
    ]


def executar_ga(
    sudoku_original,
    tamanho_populacao=60,
    tamanho_torneio=3,
    taxa_mutacao_descendente=0.20,
    taxa_mutacao_pais=0.05,
    exibir_progresso=True,
    intervalo_exibicao=100
):
    """
    Executa o Algoritmo Genético para resolução do Sudoku.

    Etapas:
    1. geração da população inicial com blocos válidos;
    2. avaliação da população;
    3. elitismo;
    4. seleção por K-Tournament;
    5. cruzamento por subgrades 3x3;
    6. mutação do descendente;
    7. possível mutação dos pais;
    8. seleção dos melhores candidatos;
    9. repetição até fitness zero ou limite de avaliações.
    """

    # --------------------------------------------------------
    # Validação dos parâmetros
    # --------------------------------------------------------

    if tamanho_populacao < 2:
        raise ValueError(
            "O tamanho da população deve ser pelo menos 2."
        )

    if tamanho_torneio <= 0:
        raise ValueError(
            "O tamanho do torneio deve ser positivo."
        )

    if not 0 <= taxa_mutacao_descendente <= 1:
        raise ValueError(
            "A taxa de mutação do descendente deve estar "
            "entre 0 e 1."
        )

    if not 0 <= taxa_mutacao_pais <= 1:
        raise ValueError(
            "A taxa de mutação dos pais deve estar entre 0 e 1."
        )

    # --------------------------------------------------------
    # Início da medição do tempo
    # --------------------------------------------------------

    tempo_inicial = time.perf_counter()

    # --------------------------------------------------------
    # População inicial
    # --------------------------------------------------------

    populacao = gerar_populacao_inicial_ga(
        sudoku_original,
        tamanho_populacao
    )

    fitness_populacao = avaliar_populacao_ga(
        populacao
    )

    # --------------------------------------------------------
    # Melhor indivíduo inicial
    # --------------------------------------------------------

    melhor_indice = int(
        np.argmin(fitness_populacao)
    )

    melhor_individuo = populacao[
        melhor_indice
    ].copy()

    melhor_fitness = int(
        fitness_populacao[melhor_indice]
    )

    historico_fitness = [melhor_fitness]

    geracao_final = 0
    geracao = 0

    # --------------------------------------------------------
    # Laço evolutivo
    # --------------------------------------------------------

    while AVALIACOES < MAX_AVALIACOES:

        geracao += 1
        geracao_final = geracao

        # ----------------------------------------------------
        # Elitismo
        # ----------------------------------------------------

        nova_populacao = [
            melhor_individuo.copy()
        ]

        # ----------------------------------------------------
        # Formação da nova população
        # ----------------------------------------------------

        while len(nova_populacao) < tamanho_populacao:

            # Cada conjunto de candidatos gera 5 avaliações
            # dentro de selecionar_melhores_candidatos_ga().
            #
            # Portanto, só prossegue se houver orçamento
            # suficiente para as cinco avaliações.
            if AVALIACOES + 5 > MAX_AVALIACOES:
                break

            # Seleção do primeiro pai
            pai_1 = selecao_torneio_ga(
                populacao,
                fitness_populacao,
                tamanho_torneio
            )

            # Seleção do segundo pai
            pai_2 = selecao_torneio_ga(
                populacao,
                fitness_populacao,
                tamanho_torneio
            )

            # Cruzamento por blocos
            descendente = cruzamento_por_blocos_ga(
                pai_1,
                pai_2
            )

            # Mutação do descendente
            descendente_mutado = mutacao_por_troca_ga(
                descendente,
                sudoku_original,
                taxa_mutacao_descendente
            )

            # Possível mutação dos pais
            pai_1_mutado = mutacao_por_troca_ga(
                pai_1,
                sudoku_original,
                taxa_mutacao_pais
            )

            pai_2_mutado = mutacao_por_troca_ga(
                pai_2,
                sudoku_original,
                taxa_mutacao_pais
            )

            # Candidatos
            candidatos = [
                pai_1,
                pai_2,
                pai_1_mutado,
                pai_2_mutado,
                descendente_mutado
            ]

            # Essa função chama calcular_fitness()
            # para cada um dos cinco candidatos.
            melhores_candidatos = (
                selecionar_melhores_candidatos_ga(
                    candidatos,
                    quantidade=2
                )
            )

            for candidato in melhores_candidatos:

                if len(nova_populacao) >= tamanho_populacao:
                    break

                nova_populacao.append(
                    candidato.copy()
                )

        # ----------------------------------------------------
        # Verifica se o orçamento terminou durante a formação
        # da nova população
        # ----------------------------------------------------

        if len(nova_populacao) < tamanho_populacao:
            break

        # A próxima operação avalia toda a nova população.
        # Só prossegue se houver orçamento suficiente.
        if AVALIACOES + tamanho_populacao > MAX_AVALIACOES:
            break

        # ----------------------------------------------------
        # Substituição da população anterior
        # ----------------------------------------------------

        populacao = nova_populacao

        # Avaliação da nova geração
        fitness_populacao = avaliar_populacao_ga(
            populacao
        )

        # ----------------------------------------------------
        # Melhor indivíduo da geração atual
        # ----------------------------------------------------

        melhor_indice_geracao = int(
            np.argmin(fitness_populacao)
        )

        melhor_fitness_geracao = int(
            fitness_populacao[melhor_indice_geracao]
        )

        # Atualização do melhor indivíduo global
        if melhor_fitness_geracao < melhor_fitness:

            melhor_fitness = melhor_fitness_geracao

            melhor_individuo = populacao[
                melhor_indice_geracao
            ].copy()

        # Registro do melhor fitness global
        historico_fitness.append(
            melhor_fitness
        )

        # ----------------------------------------------------
        # Critério de parada
        # ----------------------------------------------------

        if melhor_fitness == 0:
            break

    # --------------------------------------------------------
    # Finalização
    # --------------------------------------------------------

    tempo_execucao = (
        time.perf_counter() - tempo_inicial
    )

    resultado = {
        "algoritmo": "GA",
        "tabuleiro": melhor_individuo,
        "fitness": melhor_fitness,
        "geracoes": geracao_final,
        "iteracoes": geracao_final,
        "avaliacoes": AVALIACOES,
        "tempo": tempo_execucao,
        "historico": historico_fitness,
        "solucionado": melhor_fitness == 0
    }

    return resultado

In [ ]:
# ============================================================
# CÉLULA 5 — 30 EXECUÇÕES DO GA
# ============================================================

resultados_ga = []

PARAMETROS_GA = {

    # Quantidade de cromossomos
    "tamanho_populacao": 60,

    # Quantidade de indivíduos no K-Tournament
    "tamanho_torneio": 6,

    # Probabilidade de mutação do descendente
    "taxa_mutacao_descendente": 0.70,

    # Probabilidade de mutação dos pais
    "taxa_mutacao_pais": 0.2,

    # Desativado para evitar muitas linhas no Colab
    "exibir_progresso": False,
    "intervalo_exibicao": 100
}

for execucao in range(30):

    # Uma semente diferente para cada execução
    SEED_EXECUCAO = execucao

    random.seed(SEED_EXECUCAO)
    np.random.seed(SEED_EXECUCAO)

    # Reinicia o contador global de avaliações
    AVALIACOES = 0

    resultado = executar_ga(
        sudoku_original=sudoku,
        **PARAMETROS_GA
    )

    # Armazena os resultados da execução
    resultados_ga.append({
        "execucao": execucao + 1,
        "seed": SEED_EXECUCAO,
        "fitness": resultado["fitness"],
        "geracoes": resultado["geracoes"],
        "avaliacoes": resultado["avaliacoes"],
        "tempo": resultado["tempo"],
        "solucionado": resultado["solucionado"]
    })

    # Exibe apenas um resumo de cada execução
    print(
        f"Execução {execucao + 1:02d} | "
        f"Seed: {SEED_EXECUCAO:02d} | "
        f"Fitness: {resultado['fitness']} | "
        f"Iterações: {resultado['iteracoes']} | "
        f"Avaliações: {resultado['avaliacoes']} | "
        f"Tempo: {resultado['tempo']:.2f}s | "
        f"Solucionado: {resultado['solucionado']}"
    )

Execução 01 | Seed: 00 | Fitness: 14 | Iterações: 1905 | Avaliações: 400000 | Tempo: 98.74s | Solucionado: False
Execução 02 | Seed: 01 | Fitness: 8 | Iterações: 1905 | Avaliações: 400000 | Tempo: 99.06s | Solucionado: False
Execução 03 | Seed: 02 | Fitness: 12 | Iterações: 1905 | Avaliações: 400000 | Tempo: 98.31s | Solucionado: False
Execução 04 | Seed: 03 | Fitness: 9 | Iterações: 1905 | Avaliações: 400000 | Tempo: 98.02s | Solucionado: False
Execução 05 | Seed: 04 | Fitness: 5 | Iterações: 1905 | Avaliações: 400000 | Tempo: 98.82s | Solucionado: False
Execução 06 | Seed: 05 | Fitness: 8 | Iterações: 1905 | Avaliações: 400000 | Tempo: 97.89s | Solucionado: False
Execução 07 | Seed: 06 | Fitness: 12 | Iterações: 1905 | Avaliações: 400000 | Tempo: 99.80s | Solucionado: False
Execução 08 | Seed: 07 | Fitness: 10 | Iterações: 1905 | Avaliações: 400000 | Tempo: 98.30s | Solucionado: False
Execução 09 | Seed: 08 | Fitness: 16 | Iterações: 1905 | Avaliações: 400000 | Tempo: 99.37s | Soluci

In [ ]:
# ============================================================
# RESUMO DAS 30 EXECUÇÕES DO GA
# ============================================================

fitness = [r["fitness"] for r in resultados_ga]
tempos = [r["tempo"] for r in resultados_ga]
avaliacoes = [r["avaliacoes"] for r in resultados_ga]
geracoes = [r["geracoes"] for r in resultados_ga]

sucessos = sum(r["solucionado"] for r in resultados_ga)

taxa_sucesso = (sucessos / len(resultados_ga)) * 100

print("\n" + "=" * 50)
print("RESUMO — GA")
print("=" * 50)

print(f"Execuções: {len(resultados_ga)}")
print(f"Sucessos: {sucessos}")
print(f"Taxa de sucesso: {taxa_sucesso:.2f}%")

print(f"\nMelhor fitness: {min(fitness)}")
print(f"Pior fitness: {max(fitness)}")
print(f"Fitness médio: {np.mean(fitness):.2f}")
print(f"Mediana do fitness: {np.median(fitness):.2f}")
print(f"Desvio padrão do fitness: {np.std(fitness):.2f}")

print(f"\nTempo médio: {np.mean(tempos):.2f} segundos")
print(f"Mediana do tempo: {np.median(tempos):.2f} segundos")

print(f"\nMédia de avaliações: {np.mean(avaliacoes):.2f}")
print(f"Média de gerações: {np.mean(geracoes):.2f}")


RESUMO — GA
Execuções: 30
Sucessos: 0
Taxa de sucesso: 0.00%

Melhor fitness: 4
Pior fitness: 16
Fitness médio: 10.20
Mediana do fitness: 10.00
Desvio padrão do fitness: 2.70

Tempo médio: 168.72 segundos
Mediana do tempo: 168.65 segundos

Média de avaliações: 400000.00
Média de gerações: 1905.00
